[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# *Pipelines* de infraestructura: *plan*, *apply* y estrategia de *runners*.

Este cuaderno conecta la disciplina de *Terraform* con la estrategia de *runners* del curso: *hosted* por defecto y *self-hosted* solo cuando hay una necesidad técnica real.

## Objetivos.

- Integrar `fmt`, `validate`, `plan` y `apply` en un *workflow* gobernado.
- Definir cuándo mantener *jobs* en *hosted* y cuándo moverlos a *self-hosted*.
- Aplicar controles de seguridad por entorno antes de cambios destructivos.


## Modelo operativo: de validación a cambio controlado.

Para *IaC*, este es el orden operativo recomendado del curso:

1. escribir o ajustar código *Terraform*;
2. ejecutar `fmt`, `validate` y `plan`;
3. revisar impacto por entorno;
4. aprobar explícitamente en entornos sensibles;
5. ejecutar `apply` con trazabilidad.

Separar `plan` y `apply` no es burocracia: reduce riesgo, mejora revisión y evita cambios de infraestructura sin contexto.


## Estrategia de *runners* por fase.

En la mayoría de equipos, una estrategia sana es:

- `fmt`, `validate`, `plan` en *hosted runners*;
- `apply` en *hosted* si no hay dependencias de red privada;
- `apply` en *self-hosted* solo cuando se requiere acceso a recursos internos, segmentación o controles regulatorios.

Si se usa *self-hosted*, debe ser preferentemente efímero, con limpieza post-*job*, parcheo continuo y credenciales de vida corta.


## Ejemplo alineado al repositorio: `plan`/`apply` por entorno con *OIDC*.

```yaml
name: terraform-plan-apply

on:
  workflow_dispatch:
    inputs:
      target_environment:
        type: choice
        options: [test, prod]
      run_apply:
        type: choice
        options: ["false", "true"]

permissions:
  contents: read
  id-token: write

jobs:
  terraform:
    runs-on: ubuntu-latest
    environment: ${{ inputs.target_environment }}
    steps:
      - uses: actions/checkout@v6
      - uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ vars.GCP_WORKLOAD_IDENTITY_PROVIDER }}
          service_account: ${{ vars.GCP_SERVICE_ACCOUNT }}
      - uses: hashicorp/setup-terraform@v4
      - run: terraform init -backend-config="bucket=${{ vars.TF_STATE_BUCKET }}"
      - run: terraform fmt -check -recursive
      - run: terraform validate
      - run: terraform plan -out=tfplan
      - if: ${{ inputs.run_apply == 'true' }}
        run: terraform apply -auto-approve tfplan
```

Este patrón usa *hosted runners* y *OIDC* como ruta base. El uso de *self-hosted* se reserva para casos con necesidades de red privada o control operativo adicional.


In [ ]:
fases = [

    ('fmt-validate-plan', 'hosted + oidc'),

    ('revision-y-aprobacion', 'entorno-protegido'),

    ('apply', 'hosted-por-defecto-o-self-hosted-si-hay-red-privada')

]



for fase, runner in fases:

    print(f'fase: {fase} | estrategia: {runner}')


## Puente con el resto del bloque.

Este flujo conecta construcción de artefactos, gobierno de infraestructura y despliegue seguro en *Linux*:

- el mismo criterio de endurecimiento aplica a *build* y a *IaC*;
- los entornos protegidos definen quién promueve cambios;
- la operación en *Linux* hereda la misma trazabilidad del *pipeline*.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>